# 02 · d1-3B LoRA preparation — training shell

No weights or training are launched by Run All. M3 needs explicit approval after measured GPU compatibility and zero-shot evaluation.

All workload execution must use a managed Colab NVIDIA GPU runtime. CPU-only mathematics checks run on that remote VM, never this local system.
This notebook assumes the source archive has been extracted or the exact repository revision cloned in the runtime, and the CPU package installed. See README for upload/clone commands. Local VS Code files do not automatically become Colab runtime files. No API key goes into a cell or notebook output.

In [ ]:
"""Execution policy: workloads require a managed remote NVIDIA GPU runtime.

This standard-library-only module is also embedded in notebook preflight cells,
so hardware checks work before any package or project installation.
"""
import os
from pathlib import Path
import shutil
import subprocess


GPU_SETUP_HELP = (
    'In browser Colab select Runtime > Change runtime type > Hardware accelerator '
    '> an available NVIDIA GPU, then reconnect. In VS Code select a Colab server '
    'configured with an NVIDIA GPU and reconnect this notebook to that server. '
    'Run the preflight in the same runtime used for nbconvert. '
    'Do not try to fix GPU allocation with pip or install NVIDIA drivers in Colab.'
)


def find_nvidia_smi():
    """Resolve installed driver tooling, including common paths omitted from PATH."""
    executable = shutil.which('nvidia-smi')
    if executable:
        return executable
    for candidate in ('/usr/bin/nvidia-smi', '/usr/local/nvidia/bin/nvidia-smi'):
        if Path(candidate).is_file() and os.access(candidate, os.X_OK):
            return candidate
    return None


def require_colab_gpu():
    managed = bool(os.environ.get('COLAB_RELEASE_TAG') or
                   (Path('/content').is_dir() and os.environ.get('COLAB_BACKEND_VERSION')))
    if not managed:
        raise RuntimeError('Colab-only execution: connect a managed Colab GPU kernel; do not run locally.')
    return query_gpu('Colab', GPU_SETUP_HELP)


def require_remote_gpu():
    """Operational guard, not a security attestation: environment flags are spoofable."""
    if os.environ.get('JOB_ID'):
        accelerator = os.environ.get('ACCELERATOR', '').strip()
        if accelerator.lower() in ('', 'none'):
            raise RuntimeError('Hugging Face Jobs requires an NVIDIA GPU flavor, such as t4-small.')
        gpu = query_gpu('Hugging Face Jobs', 'Select an NVIDIA GPU Jobs flavor and retry.')
        return {'provider': 'huggingface_jobs', 'job_id': os.environ['JOB_ID'],
                'accelerator': accelerator, 'gpu': gpu}
    return {'provider': 'colab', 'gpu': require_colab_gpu()}


def query_gpu(provider, help_text):
    executable = find_nvidia_smi()
    if executable is None:
        raise RuntimeError(
            'NVIDIA GPU tooling is unavailable: nvidia-smi was not found in PATH or '
            'common driver locations. This session may be CPU/TPU or lack an attached '
            'NVIDIA GPU; no Python dependency installation has been attempted by this check. '
            + help_text
        )
    try:
        result = subprocess.run(
            [executable, '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=15,
        )
    except (OSError, subprocess.TimeoutExpired) as exc:
        raise RuntimeError(f'Could not query the {provider} NVIDIA GPU. ' + help_text) from exc
    if result.returncode or not result.stdout.strip():
        raise RuntimeError('nvidia-smi did not report an available NVIDIA GPU. ' + help_text)
    return result.stdout.strip()

import sys
print(require_colab_gpu())
REPO = Path(os.environ.get("LIQUID_MCDM_REPO", str(Path.cwd()))).resolve()
if not (REPO / "pyproject.toml").exists() and (REPO.parent / "pyproject.toml").exists():
    REPO = REPO.parent
assert (REPO / "pyproject.toml").exists(), "Upload/extract colab-source.zip or clone the repo, then set LIQUID_MCDM_REPO. See README."
os.chdir(REPO)
print("Project:", REPO)


In [ ]:
import yaml
import json
from liquid_mcdm.environment import environment_report
cfg = yaml.safe_load((REPO / "configs/d1_3b_lora.yaml").read_text())
print(json.dumps(cfg, indent=2))
print(json.dumps(environment_report(), indent=2))

**Candidate dependencies.** Start a fresh runtime and inspect `pip install --dry-run -r requirements-colab-candidate.txt`. Keep matching Torch/torchvision/CUDA. Only then install, restart the kernel, run `pip check`, record `pip freeze`, and verify model-class imports. This candidate file has not been GPU-tested. The optional install cell is disabled by default.

In [ ]:
INSTALL_CANDIDATE_DEPENDENCIES = False
if INSTALL_CANDIDATE_DEPENDENCIES:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab-candidate.txt"], check=True)
    print("Restart the kernel, rerun inventory and pip check before continuing.")

**Tokenizer-only preflight:** downloads only pinned tokenizer/config/template files, never model weights. Uses the audited upstream prompt renderer, full-precision values, alias grouping, answer/EOS boundary assertions, and reference prompt masks. This is not execution of the TRL collator.

In [ ]:
RUN_TOKENIZER_CHECK = False
if RUN_TOKENIZER_CHECK:
    from liquid_mcdm.training.preflight import tokenizer_preflight
    result = tokenizer_preflight(REPO / "outputs/smoke-v1", REPO / ".cache/d1-tokenizer")
    (REPO / "outputs/tokenizer_preflight.json").write_text(json.dumps(result, indent=2))
    print(result)
else:
    print("Tokenizer preflight is opt-in; historical result (before final edits) is reports/pre_colab_only_snapshot/tokenizer_preflight.json.")

## Explicit gate before model work

1. Obtain approval to download the pinned base checkpoint and run a bounded GPU compatibility check. Record device, free VRAM, resolved dependencies and cost/time budget.
2. Download to a revision-pinned local snapshot. Standard training loader: `AutoModelForImageTextToText`; custom inference loader: `AutoModel(trust_remote_code=True)`. Local snapshot loading avoids the custom wrapper silently resolving a tokenizer from main.
3. Inspect real `SFTTrainer` collated labels: prompt/pad `-100`; alias and EOS supervised. `completion_only_loss=True`, `packing=False`, no truncation of alternatives. Verify LoRA trainable names and frozen vision tower. Run a bounded forward/backward and standard/custom option-readout parity check.
4. Record zero-shot results on the untouched test split **before** training. Use max over alias/leading-space token groups then restricted-option softmax.
5. Only after these pass, approve the 64-example / 25-step overfit candidate in `configs/d1_3b_lora.yaml`. Use TRL+PEFT, `report_to="wandb"`, explicit offline/online mode and persisted checkpoints. Verify adapter save/reload predictions before a larger run.

The exact upstream [TRL notebook](https://github.com/Liquid4All/cookbook/blob/744c5b628a27adb4f219ff4a1d11e3d70259b369/finetuning/notebooks/sft_for_decision_models_with_trl.ipynb) supplies the model/trainer recipe. This repository intentionally stops at the tested CPU adapter and this preparation shell; an executable GPU trainer is the next M3 increment, after runtime compatibility is measured.

**Resume:** persist checkpoint, config, source revision, dataset manifest and environment freeze in a user-selected Drive folder or approved artifact store. On a fresh runtime recreate the locked environment/base snapshot and pass the verified checkpoint to `trainer.train(resume_from_checkpoint=...)`. Mounting Drive does not itself save local checkpoints; copy them periodically or use the persistent path. Never assume an adapter-only directory includes optimizer/scheduler state.